# NOTA BENE

The cox2db.csv, primary csv file with the data, was not uploaded to the repository. The cleaned up csv was.

The notebook is provided as a protocol on how the data was cleaned, in case you want to rerun it - [download the dataset here](https://www.ebi.ac.uk/chembl/explore/activities/eyJkYXRhc2V0Ijp7ImVudGl0eUlEIjoiQWN0aXZpdHkiLCJpbml0aWFsUXVlcnkiOnsicXVlcnkiOnsiYm9vbCI6eyJtdXN0IjpbeyJxdWVyeV9zdHJpbmciOnsicXVlcnkiOiJ0YXJnZXRfY2hlbWJsX2lkOkNIRU1CTDIzMCJ9fV19fX0sImZhY2V0c1N0YXRlIjpudWxsLCJjdXN0b21GaWx0ZXJpbmciOiJ0YXJnZXRfY2hlbWJsX2lkOkNIRU1CTDIzMCIsInN1YnNldEh1bWFuRGVzY3JpcHRpb24iOiJCaW9hY3Rpdml0eSBkYXRhIGZvciB0YXJnZXQgQ0hFTUJMMjMwIChQcm9zdGFnbGFuZGluIEcvSCBzeW50aGFzZSAyKSIsImV4YWN0VGV4dEZpbHRlcnMiOnt9fX0%3D) and then save it as cox2db.csv

In [1]:
import pandas as pd

# loading the df from csv and leaving only the stuff that is useful
df = pd.read_csv('cox2db.csv', sep=None, engine="python", on_bad_lines="warn")
df = df[["Molecule ChEMBL ID","Compound Key","Smiles","Standard Type","pChEMBL Value"]]

In [2]:
mask = df[df["Standard Type"].isin(["IC50", "Ki"])] # that these 2 activity types can be used together
# and other types are kind of unrelated so starting with leaving only needed ones


# here we manage what to do with the duplicates - for pchembl value we choose median cause it is important to not get too high or too
# low of a result
df = (
    df.groupby("Molecule ChEMBL ID", as_index=True)
      .agg({
          "Smiles": "first",
          "pChEMBL Value": "median",
      })
)

# sorting by activity value which is pchembl - for convenience
df = df.sort_values(by="pChEMBL Value", ascending=False)

In [3]:
df_binary = df.copy(deep=True).dropna(subset=["pChEMBL Value", "Smiles"])
# having a copy is not really needed but it just feels kind of more secure to work with it
# this way since we change the values of one column

In [4]:
import numpy as np

# IMPORTANT here the threshold for the binary classes is set to be 6, i am not sure if that is a proper one, it looked ok cause
# overall we have activities from 4 to 10 so 6 is a nice choice to say "yes whatever has higher value probably is more active"

df_binary["pChEMBL Value"] = np.where(df_binary["pChEMBL Value"] >= 6, 1, 0)
df_binary = df_binary.rename(columns={"pChEMBL Value": "Activity"})

In [5]:
print(f'{(df_binary["Activity"] == 1).sum()} - actives')
print(f'{(df_binary["Activity"] == 0).sum()} - inactives')
# classes seem to be quite balanced

2492 - actives
2028 - inactives


In [6]:
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

# here one more column is added - scaffold for each compound
# it is needed for a proper train-test splitting:
# compounds sharing a scaffold are very similar, so if we have molecules with the same scaffold in both train and test, it might
# lead to data leakage. it would be cool to try both splits with scaffolds in account and not in account to see how exactly that
# affects the result in this case. overall is considered more "honest" to use scaffold split

df_binary["scaffold"] = df_binary["Smiles"].apply(
    lambda s: MurckoScaffold.MurckoScaffoldSmiles(mol=Chem.MolFromSmiles(s))
)

df_binary.to_csv('cox2_cleaned.csv')